# Knowledge Distillation: Llama-3.1-8B → Llama-3.2-1B
**CS 455 Term Project** — Ali Kumral, Revna Demirkale

---
## ⚠️ MUST RUN FIRST — run the cell below after every runtime restart
Everything else in this notebook depends on it. It is safe to re-run at any time.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# MUST RUN FIRST — run after every runtime restart before anything else
# ═══════════════════════════════════════════════════════════════════════
import os, sys

REPO_URL   = "https://github.com/alikumral/Knowledge-Distillation-Transferring-Mathematical-Reasoning"
REPO_NAME  = "Knowledge-Distillation-Transferring-Mathematical-Reasoning"
REPO_PATH  = f"/content/{REPO_NAME}"
DRIVE_HOME = "/content/drive/MyDrive/CS455/TermProject/Knowledge-Distillation-Transferring-Mathematical-Reasoning"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not os.path.exists(REPO_PATH):
    print("Cloning repo...")
    os.system(f"git clone {REPO_URL}")
else:
    print("Repo exists, pulling latest...")
    os.system(f"git -C {REPO_PATH} pull")
    # Clear stale module cache so pulled changes take effect in this kernel.
    # (git pull updates files on disk; without this, Python keeps using the
    #  old in-memory version of every mathdistill module.)
    for _k in list(sys.modules.keys()):
        if "mathdistill" in _k:
            del sys.modules[_k]
    print("Module cache cleared.")

os.chdir(REPO_PATH)
if os.path.join(REPO_PATH, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_PATH, "src"))

os.system("pip install -q -e . -r requirements.txt")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
os.environ["MATHDISTILL_HOME"] = DRIVE_HOME
os.makedirs(DRIVE_HOME, exist_ok=True)

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)

import torch
gpu  = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠ NO GPU"
vram = f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if torch.cuda.is_available() else ""
print(f"\n{'='*55}")
print(f"  GPU:       {gpu}  {vram}")
print(f"  Repo:      {os.getcwd()}")
print(f"  Artifacts: {os.environ['MATHDISTILL_HOME']}")
print(f"{'='*55}")
print("  Setup complete. Ready to run any stage below.")
print(f"{'='*55}")

---
## [OPTIONAL] Smoke test — already passed, skip this
⚠️ **Do NOT run before Stage 1.** Loading both models dirties VRAM even after `del model`.

In [ ]:
# OPTIONAL — already passed on 2026-05-29. Skip before Stage 1.
import torch
from mathdistill.utils import load_config
from mathdistill.models import load_model_4bit, load_tokenizer, render_prompt
from mathdistill.prompts import build_messages
from mathdistill.answers import extract_pred_number

tcfg = load_config("configs/teacher_gen.yaml")
q = ("Natalia sold clips to 48 friends in April, and half as many in May. "
     "How many clips did she sell altogether?")

for model_id in [tcfg["model_id"], "meta-llama/Llama-3.2-1B-Instruct"]:
    print("=" * 60, "\nLoading", model_id)
    tok = load_tokenizer(model_id)
    model = load_model_4bit(model_id, tcfg["quantization"])
    enc = tok(render_prompt(tok, build_messages(q)), return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=256, do_sample=False, pad_token_id=tok.pad_token_id)
    text = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    print(text)
    print(">> extracted:", extract_pred_number(text), "(expected 72)")
    del model
    torch.cuda.empty_cache()

---
## Stage 1 — Teacher CoT Generation

**Strategy:** 3 passes × 1 sample/problem × batch_size=16 ≈ **14h per pass** (vs. 110h for the original k=4, batch_size=4 design).

| Pass | `PASS =` | sample_idx written | Expected time | Status |
|---|---|---|---|---|
| 0 | `0` | 0 for all 7,473 problems | ~14h | ✓ pilot done (30 problems) |
| 1 | `1` | 1 for all 7,473 problems | ~14h | pending |
| 2 | `2` | 2 for all 7,473 problems | ~14h | pending |

**Each pass is independent and resumable** — if Colab disconnects, re-run the MUST-RUN-FIRST cell and this cell with the same `PASS` value. It picks up where it left off.

After all 3 passes: run Stage 2 to build SFT datasets.

In [ ]:
from mathdistill.generate import generate_teacher_traces
from mathdistill.utils import load_config

PILOT = False  # True = 30 problems (~3 min to validate). False = full 7,473 problems.
PASS  = 0      # ← change to 0, 1, or 2 for each separate generation pass

cfg = load_config("configs/teacher_gen.yaml")
cfg["pass_id"] = PASS
if PILOT:
    cfg["limit"] = 30

print(f"Starting pass {PASS} | pilot={PILOT} | batch_size={cfg['batch_size']} | k={cfg['generation']['k_samples']}")
generate_teacher_traces(cfg, load_config("configs/data.yaml"))

---
## Stage 2 — Rejection Sampling & Build SFT Datasets

**Run after each pass** to check acceptance stats (Risk-1 check).  
**Run `build_all_sft_datasets`** only after all 3 passes are complete.

Expected after 3 passes: ~6,700 problems covered (90% of 7,473), ~20k total traces.

In [ ]:
import json
from mathdistill.reject import rejection_sample, acceptance_stats, build_all_sft_datasets
from mathdistill.utils import artifact_path, load_config

traces_path   = artifact_path("traces", "teacher_traces.jsonl")
accepted_path = artifact_path("sft", "accepted.jsonl")

rejection_sample(traces_path, accepted_path)
stats = acceptance_stats(accepted_path)
print(json.dumps(stats, indent=2))

if stats["n_problems_with_ge1"] < 18:  # <60% of 30-problem pilot
    print("\n⚠ Low acceptance — consider Plan B (more samples or swap teacher)")
else:
    print("\n✓ Acceptance looks good")

# ── Uncomment ONLY after all 3 passes are complete ─────────────────────
# build_all_sft_datasets(accepted_path, load_config("configs/data.yaml"))
# print("SFT datasets written to Drive.")

---
## Stage 3 — QLoRA Fine-tuning
Change `CONFIG` and `SEED` per run (~2h each). Spread across sessions.

In [ ]:
# Clear any stale mathdistill imports before loading (safe to run multiple times)
import sys
for _k in list(sys.modules.keys()):
    if "mathdistill" in _k:
        del sys.modules[_k]

from mathdistill.train import train_one
from mathdistill.utils import load_config

CONFIG = "configs/train_teacher_1.yaml"   # train_teacher_1 | train_teacher_3 | train_gold
SEED   = 0                                # 0, 1, 2

train_one(load_config(CONFIG), load_config("configs/data.yaml"), seed=SEED)

---
## Stage 4 — Evaluation
Greedy decoding on GSM8K test + MATH-200. Accuracy with 95% bootstrap CIs.

In [ ]:
import sys
for _k in list(sys.modules.keys()):
    if "mathdistill" in _k:
        del sys.modules[_k]

STUDENT = "meta-llama/Llama-3.2-1B-Instruct"
TEACHER = "meta-llama/Llama-3.1-8B-Instruct"

# Each run_eval.py call is a separate subprocess — model memory is freed between conditions.
# Run them one at a time; comment out conditions you haven't trained yet.

# (i) zero-shot 1B — no adapter
!python scripts/run_eval.py --model-id {STUDENT} --name zeroshot

# (ii) gold CoT, seed 0
!python scripts/run_eval.py --model-id {STUDENT} --adapter adapters/gold/seed0 --name gold_seed0

# (iii) teacher_1, seed 0
!python scripts/run_eval.py --model-id {STUDENT} --adapter adapters/teacher_1/seed0 --name teacher_1_seed0

# (iv) teacher_3, seed 0
!python scripts/run_eval.py --model-id {STUDENT} --adapter adapters/teacher_3/seed0 --name teacher_3_seed0

# (v) 8B teacher upper bound — takes longer (~3x eval time)
!python scripts/run_eval.py --model-id {TEACHER} --name teacher

In [ ]:
import glob, os, pandas as pd
from mathdistill.utils import read_jsonl
from mathdistill.metrics import accuracy, bootstrap_ci

pred_dir = os.path.join(os.environ["MATHDISTILL_HOME"], "results", "predictions")
rows = []
for path in sorted(glob.glob(os.path.join(pred_dir, "*.jsonl"))):
    data = read_jsonl(path)
    correct = [r["correct"] for r in data]
    lo, hi = bootstrap_ci(correct)
    rows.append({"condition": os.path.basename(path).replace(".jsonl", ""),
                 "n": len(correct), "acc": round(accuracy(correct), 4),
                 "ci_lo": round(lo, 4), "ci_hi": round(hi, 4)})
pd.DataFrame(rows)